# Hindi (en-hi) two-stage fine-tune on Colab GPU

Before running: **Runtime -> Change runtime type -> Hardware accelerator -> GPU** -> Save.

Two-stage fine-tune, not one: **Stage 1** adapts the pretrained model to Hindi generally (98,000 pairs, IITB+BPCC, subsampled from a 1.58M-pair pool to keep training time in the same ballpark as the Mandarin run rather than the ~11h full-pool estimate). **Stage 2** takes Stage 1's *output* checkpoint and specializes it on pooled medical data (3,949 pairs: TICO-19 + Lokmat + Zenodo). Run both cells in order -- Stage 2 will fail loudly if Stage 1 hasn't produced a `final` checkpoint yet.

No GitHub access needed -- everything is written directly to the Colab VM via `%%writefile`, same as the Mandarin notebook.

**Manual step, do this first**: upload these files to Drive (paths shown are local repo paths -> where to put them):
- `data/processed/en-hi-general/{train,val}.{en,hi}` -> `MyDrive/medical_vlm_data/en-hi-general/`
- `data/processed/en-hi-medical/{train,val}.{en,hi}` -> `MyDrive/medical_vlm_data/en-hi-medical/`
- `data/eval/tico19_hi/{tico19.en,tico19.hi}` -> `MyDrive/medical_vlm_data/eval/tico19_hi/`

Checkpoints persist to Drive (`models/` is symlinked there), same resume-after-disconnect design as the Mandarin notebook: every cell that needs the working directory sets it explicitly, and both training cells auto-detect an existing checkpoint and resume instead of restarting.

The Hindi **baseline** (pretrained, not-yet-fine-tuned) is already done locally on CPU (~4 min, see `results/baseline_en-hi.json`: LaBSE 0.807, BLEU 13.5, chrF 34.4) -- nothing to redo here, this notebook is fine-tuning + a fine-tuned re-eval only.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only -- stop and pick a GPU runtime')
assert torch.cuda.is_available(), 'No GPU attached -- use Runtime -> Change runtime type -> GPU'

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,memory.free --format=csv

In [ ]:
import os

REPO_DIR = "/content/repo"
os.makedirs(f"{REPO_DIR}/src", exist_ok=True)
os.makedirs(f"{REPO_DIR}/configs", exist_ok=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
%%writefile src/train.py
"""Fine-tune a pretrained MarianMT checkpoint on a medical parallel corpus.

Mirrors teammate 2's per-direction MarianMT fine-tuning pattern: one pretrained
Helsinki-NLP/opus-mt-* checkpoint in, one fine-tuned checkpoint out, driven by
a small YAML config so the same script covers every direction (en-zh, zh-en,
en-hi, hi-en, ...) with only the config changed. Also doubles as the Stage-2
script for a two-stage fine-tune: `base_model` in the config can point at a
local checkpoint directory (Stage 1's output) just as easily as a HF hub id.

Usage:
    python src/train.py --config configs/en-zh.yaml
"""
import argparse
from pathlib import Path

import numpy as np
import torch
import yaml
from datasets import Dataset
from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    MarianMTModel,
    MarianTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)


def load_parallel_file_pair(src_path: Path, tgt_path: Path) -> Dataset:
    with src_path.open(encoding="utf-8") as f:
        src_lines = [l.rstrip("\n") for l in f]
    with tgt_path.open(encoding="utf-8") as f:
        tgt_lines = [l.rstrip("\n") for l in f]
    assert len(src_lines) == len(tgt_lines), (
        f"{src_path} has {len(src_lines)} lines, {tgt_path} has {len(tgt_lines)}"
    )
    return Dataset.from_dict({"src": src_lines, "tgt": tgt_lines})


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", required=True, type=Path)
    ap.add_argument("--resume-from-checkpoint", default=None)
    args = ap.parse_args()

    cfg = yaml.safe_load(args.config.read_text(encoding="utf-8"))

    base_model = cfg["base_model"]
    output_dir = Path(cfg["output_dir"])
    max_src_len = cfg.get("max_source_length", 128)
    max_tgt_len = cfg.get("max_target_length", 128)

    print(f"Loading base model {base_model} ...")
    tokenizer = MarianTokenizer.from_pretrained(base_model)
    model = MarianMTModel.from_pretrained(base_model)

    train_ds = load_parallel_file_pair(Path(cfg["train_src"]), Path(cfg["train_tgt"]))
    val_ds = load_parallel_file_pair(Path(cfg["val_src"]), Path(cfg["val_tgt"]))
    print(f"Train examples: {len(train_ds)} | Val examples: {len(val_ds)}")

    def preprocess(batch):
        model_inputs = tokenizer(
            batch["src"], max_length=max_src_len, truncation=True,
        )
        labels = tokenizer(
            text_target=batch["tgt"], max_length=max_tgt_len, truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    train_tok = train_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])
    val_tok = val_ds.map(preprocess, batched=True, remove_columns=["src", "tgt"])

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

    def compute_metrics(eval_preds):
        import sacrebleu

        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.corpus_bleu(decoded_preds, [decoded_labels])
        return {"bleu": bleu.score}

    training_args_kwargs = dict(
        output_dir=str(output_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        learning_rate=cfg.get("learning_rate", 2e-5),
        per_device_train_batch_size=cfg.get("per_device_train_batch_size", 16),
        per_device_eval_batch_size=cfg.get("per_device_eval_batch_size", 16),
        weight_decay=cfg.get("weight_decay", 0.01),
        num_train_epochs=cfg.get("num_train_epochs", 3),
        predict_with_generate=True,
        generation_max_length=max_tgt_len,
        fp16=torch.cuda.is_available(),
        load_best_model_at_end=True,
        metric_for_best_model="bleu",
        greater_is_better=True,
        save_total_limit=cfg.get("save_total_limit", 2),
        report_to=cfg.get("report_to", []),
        group_by_length=cfg.get("group_by_length", False),
        dataloader_num_workers=cfg.get("dataloader_num_workers", 0),
    )
    # Filter against whatever this installed transformers version actually
    # accepts -- API surface (e.g. field renames) drifts across major
    # versions, and this script needs to run unmodified on both a pinned
    # local install and whatever Colab happens to have pulled in.
    import dataclasses
    valid_fields = {f.name for f in dataclasses.fields(Seq2SeqTrainingArguments)}
    dropped = {k: v for k, v in training_args_kwargs.items() if k not in valid_fields}
    if dropped:
        print(f"NOTE: this transformers install doesn't support {list(dropped)} -- skipping (had no effect anyway if unsupported)")
    training_args_kwargs = {k: v for k, v in training_args_kwargs.items() if k in valid_fields}

    training_args = Seq2SeqTrainingArguments(**training_args_kwargs)

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        data_collator=data_collator,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=cfg.get("early_stopping_patience", 3))],
    )

    trainer.train(resume_from_checkpoint=args.resume_from_checkpoint)

    final_dir = output_dir / "final"
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(str(final_dir))
    print(f"Saved fine-tuned model to {final_dir}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/en-hi-stage1-general-colab-gpu.yaml
# Hindi two-stage fine-tune, Stage 1: general-domain adaptation.
# Subsampled to 98,000 train pairs (from a 1,581,035-pair pool of IITB+BPCC)
# via `data_prep.py --max-pairs 100000` specifically to keep training time in
# the same ballpark as the Mandarin GPU run (~27 min for 63,831 pairs) rather
# than the ~11 hour full-pool estimate -- a deliberate scope/time trade-off,
# not an oversight. Composition after the cap: ~90% iitb, ~10% bpcc_ilci
# (proportional to the pre-cap pool, since the cap is a random subsample).

base_model: Helsinki-NLP/opus-mt-en-hi
output_dir: models/opus-mt-en-hi-stage1-general

train_src: data/processed/en-hi-general/train.en
train_tgt: data/processed/en-hi-general/train.hi
val_src: data/processed/en-hi-general/val.en
val_tgt: data/processed/en-hi-general/val.hi

max_source_length: 128
max_target_length: 128

num_train_epochs: 3
learning_rate: 2.0e-5

per_device_train_batch_size: 64
per_device_eval_batch_size: 64
group_by_length: true
dataloader_num_workers: 2

weight_decay: 0.01
early_stopping_patience: 3
save_total_limit: 2
report_to: []

In [ ]:
%%writefile configs/en-hi-stage2-medical-colab-gpu.yaml
# Hindi two-stage fine-tune, Stage 2: medical specialization.
# base_model points at Stage 1's OUTPUT checkpoint, not the raw pretrained
# model -- this is what makes it a two-stage fine-tune. Run Stage 1 first.
#
# Only 3,949 train pairs (tico19 + lokmat_healthcare + zenodo_healthcare,
# with 33 untranslated-in-source rows filtered out) -- small enough that
# more epochs than usual are worth it, relying on early_stopping_patience
# to cut it short once eval BLEU stops improving.

base_model: models/opus-mt-en-hi-stage1-general/final
output_dir: models/opus-mt-en-hi-medical

train_src: data/processed/en-hi-medical/train.en
train_tgt: data/processed/en-hi-medical/train.hi
val_src: data/processed/en-hi-medical/val.en
val_tgt: data/processed/en-hi-medical/val.hi

max_source_length: 128
max_target_length: 128

num_train_epochs: 10
learning_rate: 2.0e-5

per_device_train_batch_size: 32
per_device_eval_batch_size: 32
group_by_length: true
dataloader_num_workers: 2

weight_decay: 0.01
early_stopping_patience: 3
save_total_limit: 2
report_to: []

In [ ]:
# Deliberately NOT reinstalling torch -- Colab's preinstalled torch already has
# CUDA wired up correctly; reinstalling from requirements.txt risks silently
# swapping it for a CPU-only build.
!pip install -q transformers sentencepiece sacrebleu PyYAML accelerate sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil

os.chdir("/content/repo")

DRIVE_GENERAL_DIR = "/content/drive/MyDrive/medical_vlm_data/en-hi-general"
DRIVE_MEDICAL_DIR = "/content/drive/MyDrive/medical_vlm_data/en-hi-medical"

for drive_dir, local_dir, files in [
    (DRIVE_GENERAL_DIR, "data/processed/en-hi-general", ["train.en", "train.hi", "val.en", "val.hi"]),
    (DRIVE_MEDICAL_DIR, "data/processed/en-hi-medical", ["train.en", "train.hi", "val.en", "val.hi"]),
]:
    assert os.path.isdir(drive_dir), f"{drive_dir} not found -- upload the processed data there first"
    os.makedirs(local_dir, exist_ok=True)
    for f in files:
        src = os.path.join(drive_dir, f)
        assert os.path.isfile(src), f"missing {f} in {drive_dir}"
        shutil.copy(src, os.path.join(local_dir, f))

# models/ -> Drive, so checkpoints from BOTH stages (and the final models) survive a disconnect
DRIVE_CHECKPOINT_ROOT = "/content/drive/MyDrive/medical_vlm_checkpoints"
os.makedirs(DRIVE_CHECKPOINT_ROOT, exist_ok=True)
if os.path.islink("models") or os.path.isdir("models"):
    if os.path.islink("models"):
        os.remove("models")
    else:
        shutil.rmtree("models")
os.symlink(DRIVE_CHECKPOINT_ROOT, "models")
print("Data and checkpoint dir ready.")

In [ ]:
import os, glob

os.chdir("/content/repo")

STAGE1_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-hi-stage1-general"
checkpoints = sorted(glob.glob(os.path.join(STAGE1_CHECKPOINT_DIR, "checkpoint-*")), key=os.path.getmtime)
resume_flag = f'--resume-from-checkpoint "{checkpoints[-1]}"' if checkpoints else ""
if checkpoints:
    print(f"Resuming Stage 1 from {checkpoints[-1]}")
else:
    print("No existing Stage 1 checkpoint found -- starting fresh.")

!python src/train.py --config configs/en-hi-stage1-general-colab-gpu.yaml {resume_flag}

In [ ]:
import os, glob

os.chdir("/content/repo")

assert os.path.isdir("models/opus-mt-en-hi-stage1-general/final"), (
    "Stage 1 hasn't finished (no models/opus-mt-en-hi-stage1-general/final) -- "
    "run the Stage 1 cell above to completion before Stage 2"
)

STAGE2_CHECKPOINT_DIR = "/content/drive/MyDrive/medical_vlm_checkpoints/opus-mt-en-hi-medical"
checkpoints = sorted(glob.glob(os.path.join(STAGE2_CHECKPOINT_DIR, "checkpoint-*")), key=os.path.getmtime)
resume_flag = f'--resume-from-checkpoint "{checkpoints[-1]}"' if checkpoints else ""
if checkpoints:
    print(f"Resuming Stage 2 from {checkpoints[-1]}")
else:
    print("No existing Stage 2 checkpoint found -- starting fresh from Stage 1's output.")

!python src/train.py --config configs/en-hi-stage2-medical-colab-gpu.yaml {resume_flag}

# Post-training verification

Same two checks as the Mandarin notebook: a fast sanity check that the saved Stage 2 checkpoint actually loads and translates sensibly (real sentences from the real held-out TICO-19 set, not invented), then the real before/after comparison against the already-measured baseline (LaBSE 0.807, BLEU 13.5, chrF 34.4).

**One more manual step**: upload `data/eval/tico19_hi/{tico19.en,tico19.hi}` to `MyDrive/medical_vlm_data/eval/tico19_hi/` in Drive.

In [ ]:
import os, shutil

os.chdir("/content/repo")

DRIVE_EVAL_DIR = "/content/drive/MyDrive/medical_vlm_data/eval/tico19_hi"
assert os.path.isdir(DRIVE_EVAL_DIR), f"{DRIVE_EVAL_DIR} not found -- upload tico19.{{en,hi}} there first"
for f in ["tico19.en", "tico19.hi"]:
    assert os.path.isfile(os.path.join(DRIVE_EVAL_DIR, f)), f"missing {f} in {DRIVE_EVAL_DIR}"

os.makedirs("data/eval/tico19_hi", exist_ok=True)
for f in ["tico19.en", "tico19.hi"]:
    shutil.copy(os.path.join(DRIVE_EVAL_DIR, f), os.path.join("data/eval/tico19_hi", f))

print("Eval data ready:", os.listdir("data/eval/tico19_hi"))

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer

with open("data/eval/tico19_hi/tico19.en", encoding="utf-8") as f:
    en_lines = f.read().splitlines()
with open("data/eval/tico19_hi/tico19.hi", encoding="utf-8") as f:
    hi_lines = f.read().splitlines()

# Real sentences, real indices (all 9 of them -- the full pilot-term-matched
# set), pulled directly from the baseline run's actual results
# (results/baseline_en-hi.json), not invented. This is the exact subset that
# scored worst at baseline (mean LaBSE 0.645 vs 0.807 overall), so it's the
# most direct check of whether Stage 2 fixed the technical-pathology-language
# gap the baseline struggled with.
pilot_indices = [279, 914, 941, 1330, 1900, 1930, 1933, 1935, 1973]

print(f"Loading fine-tuned checkpoint on {'cuda' if torch.cuda.is_available() else 'cpu'} ...")
tokenizer = MarianTokenizer.from_pretrained("models/opus-mt-en-hi-medical/final")
model = MarianMTModel.from_pretrained("models/opus-mt-en-hi-medical/final")
if torch.cuda.is_available():
    model = model.cuda()
model.eval()

for i in pilot_indices:
    en, ref = en_lines[i], hi_lines[i]
    inputs = tokenizer([en], return_tensors="pt", truncation=True, max_length=128)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_length=128)
    hyp = tokenizer.batch_decode(out, skip_special_tokens=True)[0]
    print(f"\n[test_split_index {i}]")
    print("en: ", en)
    print("ref:", ref)
    print("hyp:", hyp)

In [ ]:
%%writefile src/baseline_eval.py
"""Baseline evaluation of a *pretrained, not-yet-fine-tuned* MarianMT checkpoint,
scored against real held-out human-translated data already in this project.

Why this script exists and runs before any fine-tuning: fine-tuning without a
baseline means there is nothing to compare against afterward -- you'd have a
model and a feeling that it's probably better, with no number to put in a
paper. This measures the pretrained checkpoint first, so `train.py`'s output
can later be measured with the exact same script and compared directly.

*** Standing rule: never generate ground truth ***
An earlier version of this script used hand-written pilot sentences with
reference translations invented by Claude. That was wrong and has been
replaced: every source/reference pair scored here is pulled verbatim from
real, existing, human-translated data -- never written or guessed on the
spot, even as a placeholder. Inventing a reference makes the eval circular
(you'd be measuring "how similar is this to what an AI guessed," not "how
correct is this translation"), which defeats the entire purpose of a number
meant to appear in a paper. Real sources used:
  - Mandarin: `data/eval/nejm_zh/nejm.test.{en,zh}` -- NEJM-enzh's own
    test split (2,102 pairs), medical, professionally human-translated.
  - Hindi:    `data/eval/tico19_hi/tico19.{en,hi}` -- TICO-19's test split
    (2,100 pairs), medical/COVID-domain, human-translated.
Both are held-out test splits, never touched by data_prep.py's training
pool, so they stay valid for a fine-tuned-model re-run later too.

Since the pilot label set (cardiomegaly, pleural effusion, pneumothorax,
edema, "no acute cardiopulmonary abnormality", support devices,
atelectasis, consolidation) comes from chest-X-ray reporting (CheXpert/
MIMIC-CXR labels) and these two real corpora are general internal-medicine
literature / COVID guidance -- not radiology reports -- most of those exact
terms are rare or absent in the real data. Per the standing rule, that
absence is reported explicitly (see "pilot_term_coverage" in the output),
not papered over with invented sentences.

Known preprocessing fix (Mandarin only): NEJM-enzh's raw files are
Moses-style pre-tokenized. Both are de-tokenized on the fly before scoring.
A fine-tuned zh checkpoint also gets its *output* normalized before scoring
(normalize_hyp_for_scoring) since a model trained on the (formerly)
un-detokenized training pool learned to emit the same space-segmented
format -- see docs/process_guide.md for the full story. Hindi doesn't have
either issue: TICO-19/Lokmat/Zenodo were never Moses-tokenized, and Hindi
uses spaces between words natively, so normalize_hyp_for_scoring is off.

Usage:
    python src/baseline_eval.py --lang zh --model Helsinki-NLP/opus-mt-en-zh --label baseline --out results/baseline_en-zh.json
    python src/baseline_eval.py --lang hi --model Helsinki-NLP/opus-mt-en-hi --label baseline --out results/baseline_en-hi.json

    # after fine-tuning, re-run against the fine-tuned checkpoint for a direct delta:
    python src/baseline_eval.py --lang hi --model models/opus-mt-en-hi-medical/final --label fine-tuned --out results/finetuned_en-hi.json
"""
import argparse
import json
import re
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path

DEFAULT_MODEL = {
    "zh": "Helsinki-NLP/opus-mt-en-zh",
    "hi": "Helsinki-NLP/opus-mt-en-hi",
}

DATA_SOURCES = {
    "zh": {
        "src": Path("data/eval/nejm_zh/nejm.test.en"),
        "tgt": Path("data/eval/nejm_zh/nejm.test.zh"),
        "description": "NEJM-enzh test split (2,102 pairs), medical, human-translated (professional NEJM translators)",
        "detokenize_src": True,
        "detokenize_tgt": True,
        "normalize_hyp_for_scoring": True,
    },
    "hi": {
        "src": Path("data/eval/tico19_hi/tico19.en"),
        "tgt": Path("data/eval/tico19_hi/tico19.hi"),
        "description": "TICO-19 en-hi test split (2,100 pairs), medical/COVID-domain, human-translated",
        "detokenize_src": False,
        "detokenize_tgt": False,
        "normalize_hyp_for_scoring": False,
    },
}

PILOT_TERMS = [
    ("cardiomegaly", "cardiomegaly"),
    ("pleural_effusion", "pleural effusion"),
    ("pneumothorax", "pneumothorax"),
    ("edema", "edema"),
    ("no_acute_cardiopulmonary_abnormality_EXACT_PHRASE", "no acute cardiopulmonary abnormality"),
    ("cardiopulmonary_BROAD_LOOSE_MATCH", "cardiopulmonary"),
    ("support_devices", "support device"),
    ("atelectasis", "atelectasis"),
    ("consolidation", "consolidation"),
]


def detokenize_moses_en(text: str) -> str:
    text = text.replace(" @-@ ", "-")
    text = re.sub(r"\s+([,.;:!?)])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    return text.strip()


def detokenize_segmented_zh(text: str) -> str:
    return text.replace(" ", "").strip()


def load_pairs(src_path: Path, tgt_path: Path, detok_src: bool, detok_tgt: bool):
    src_lines = src_path.read_text(encoding="utf-8").splitlines()
    tgt_lines = tgt_path.read_text(encoding="utf-8").splitlines()
    assert len(src_lines) == len(tgt_lines), f"{src_path} has {len(src_lines)} lines, {tgt_path} has {len(tgt_lines)}"
    pairs = []
    for s, t in zip(src_lines, tgt_lines):
        if detok_src:
            s = detokenize_moses_en(s)
        if detok_tgt:
            t = detokenize_segmented_zh(t)
        pairs.append((s, t))
    return pairs


def find_term_matches(pairs, terms):
    matches = {label: [] for label, _ in terms}
    for i, (en, _ref) in enumerate(pairs):
        low = en.lower()
        for label, substring in terms:
            if substring in low:
                matches[label].append(i)
    return matches


def cosine_sim(a, b) -> float:
    import numpy as np
    a = np.asarray(a, dtype="float64")
    b = np.asarray(b, dtype="float64")
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--lang", required=True, choices=sorted(DATA_SOURCES), help="target language")
    ap.add_argument("--model", default=None, help=f"checkpoint to evaluate (HF hub id or local dir). Defaults per --lang: {DEFAULT_MODEL}")
    ap.add_argument("--labse-model", default="sentence-transformers/LaBSE")
    ap.add_argument("--label", default="baseline", help="free-text tag for this run, stored in the report")
    ap.add_argument("--out", required=True, type=Path)
    ap.add_argument("--max-length", type=int, default=128)
    ap.add_argument("--sample-size", type=int, default=200, help="random sample size from the non-pilot-matched remainder (0 = full remainder)")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    model_id = args.model or DEFAULT_MODEL[args.lang]
    source = DATA_SOURCES[args.lang]
    for key in ("src", "tgt"):
        if not source[key].exists():
            raise SystemExit(f"Expected real held-out data at {source[key]} but it doesn't exist.")

    print(f"Data source: {source['description']}")
    pairs = load_pairs(source["src"], source["tgt"], source["detokenize_src"], source["detokenize_tgt"])
    print(f"Loaded {len(pairs)} real held-out pairs from {source['src'].name} / {source['tgt'].name}")

    term_matches = find_term_matches(pairs, PILOT_TERMS)
    pilot_term_coverage = {label: len(idxs) for label, idxs in term_matches.items()}
    print("\nPilot term coverage in real held-out data:")
    for label, count in pilot_term_coverage.items():
        print(f"  {label}: {count}")

    matched_indices = sorted({i for idxs in term_matches.values() for i in idxs})
    matched_set = set(matched_indices)
    remaining_indices = [i for i in range(len(pairs)) if i not in matched_set]

    import random
    rng = random.Random(args.seed)
    if args.sample_size and args.sample_size > 0:
        sample_indices = rng.sample(remaining_indices, min(args.sample_size, len(remaining_indices)))
    else:
        sample_indices = remaining_indices
    selected_indices = sorted(matched_set | set(sample_indices))
    print(f"\nEvaluating {len(selected_indices)} sentences total: {len(matched_indices)} pilot-term matches + {len(sample_indices)} randomly sampled (seed={args.seed})")

    index_to_terms = {}
    for label, idxs in term_matches.items():
        for i in idxs:
            index_to_terms.setdefault(i, []).append(label)

    import torch
    from sentence_transformers import SentenceTransformer
    from transformers import MarianMTModel, MarianTokenizer
    import sacrebleu

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nDevice: {device}")
    print(f"Loading MT checkpoint: {model_id}")
    tokenizer = MarianTokenizer.from_pretrained(model_id)
    model = MarianMTModel.from_pretrained(model_id).to(device)
    model.eval()

    print(f"Loading LaBSE: {args.labse_model}")
    labse = SentenceTransformer(args.labse_model, device=device)

    per_sentence = []

    for n, i in enumerate(selected_indices, 1):
        en, ref = pairs[i]
        terms_here = index_to_terms.get(i, [])

        inputs = tokenizer([en], return_tensors="pt", truncation=True, max_length=args.max_length).to(device)
        start = time.perf_counter()
        with torch.no_grad():
            generated = model.generate(**inputs, max_length=args.max_length)
        if device == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        hyp = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
        hyp_scored = detokenize_segmented_zh(hyp) if source.get("normalize_hyp_for_scoring") else hyp

        emb = labse.encode([hyp_scored, ref], normalize_embeddings=False)
        labse_score = cosine_sim(emb[0], emb[1])
        sent_bleu = sacrebleu.sentence_bleu(hyp_scored, [ref]).score

        per_sentence.append({
            "test_split_index": i,
            "pilot_terms_matched": terms_here,
            "source_en": en,
            "reference": ref,
            "hypothesis": hyp,
            "hypothesis_scored": hyp_scored if hyp_scored != hyp else None,
            "labse_cosine": labse_score,
            "sentence_bleu": sent_bleu,
            "inference_seconds": elapsed,
        })
        if n % 25 == 0 or n == len(selected_indices):
            print(f"  [{n}/{len(selected_indices)}] {elapsed*1000:.0f}ms  LaBSE={labse_score:.3f}  BLEU={sent_bleu:.1f}" + (f"  terms={terms_here}" if terms_here else ""))

    def summarize(rows):
        if not rows:
            return None
        labse_scores = [r["labse_cosine"] for r in rows]
        timings = [r["inference_seconds"] for r in rows]
        hyps = [r["hypothesis_scored"] or r["hypothesis"] for r in rows]
        refs = [r["reference"] for r in rows]
        return {
            "n_sentences": len(rows),
            "mean_labse_cosine": statistics.mean(labse_scores),
            "min_labse_cosine": min(labse_scores),
            "corpus_bleu": sacrebleu.corpus_bleu(hyps, [refs]).score,
            "corpus_chrf": sacrebleu.corpus_chrf(hyps, [refs]).score,
            "mean_inference_seconds": statistics.mean(timings),
            "median_inference_seconds": statistics.median(timings),
            "max_inference_seconds": max(timings),
        }

    pilot_rows = [r for r in per_sentence if r["pilot_terms_matched"]]
    summary_overall = summarize(per_sentence)
    summary_pilot_subset = summarize(pilot_rows)

    report = {
        "label": args.label,
        "model": model_id,
        "lang_pair": f"en-{args.lang}",
        "device": device,
        "labse_model": args.labse_model,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "data_source": {
            "src_file": str(source["src"]),
            "tgt_file": str(source["tgt"]),
            "description": source["description"],
        },
        "reference_translation_provenance": "Real human-translated held-out test data, never generated or invented.",
        "pilot_term_coverage": pilot_term_coverage,
        "sample_size_requested": args.sample_size,
        "seed": args.seed,
        "per_sentence": per_sentence,
        "summary_overall": summary_overall,
        "summary_pilot_term_subset": summary_pilot_subset,
    }

    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

    print("\n=== Summary (overall, all evaluated sentences) ===")
    print(f"model:              {model_id}")
    print(f"n sentences:        {summary_overall['n_sentences']}")
    print(f"mean LaBSE cosine:  {summary_overall['mean_labse_cosine']:.3f}")
    print(f"corpus BLEU:        {summary_overall['corpus_bleu']:.1f}")
    print(f"corpus chrF:        {summary_overall['corpus_chrf']:.1f}")
    print(f"mean inference:     {summary_overall['mean_inference_seconds']*1000:.0f} ms/sentence")
    print(f"median inference:   {summary_overall['median_inference_seconds']*1000:.0f} ms/sentence")
    if summary_pilot_subset:
        print(f"\n=== Summary (pilot-term-matched subset only, n={summary_pilot_subset['n_sentences']}) ===")
        print(f"mean LaBSE cosine:  {summary_pilot_subset['mean_labse_cosine']:.3f}")
        print(f"corpus BLEU:        {summary_pilot_subset['corpus_bleu']:.1f}")
    else:
        print("\n(no pilot-term matches found in this real test split -- see pilot_term_coverage above)")
    print(f"\nReport saved to {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
import os, shutil

os.chdir("/content/repo")

!python src/baseline_eval.py --lang hi --model models/opus-mt-en-hi-medical/final --label fine-tuned --out results/finetuned_en-hi.json

os.makedirs("/content/drive/MyDrive/medical_vlm_checkpoints/results", exist_ok=True)
shutil.copy("results/finetuned_en-hi.json", "/content/drive/MyDrive/medical_vlm_checkpoints/results/finetuned_en-hi.json")
print("Report also copied to Drive.")

In [ ]:
import json

# Real numbers already measured (results/baseline_en-hi.json, generated before
# any fine-tuning, 2026-08-01) -- transcribed here so this cell works even if
# that file isn't present in this Colab session.
BASELINE = {
    "overall": {"n": 209, "labse": 0.8073323605742323, "bleu": 13.498593731227643, "chrf": 34.40555583242876},
    "pilot_subset": {"n": 9, "labse": 0.6448668646718256, "bleu": 2.920775524336899},
}

with open("results/finetuned_en-hi.json", encoding="utf-8") as f:
    finetuned = json.load(f)

fo, fp = finetuned["summary_overall"], finetuned["summary_pilot_term_subset"]

def fmt_delta(before, after):
    d = after - before
    return f"{before:.3f} -> {after:.3f}  ({'+' if d >= 0 else ''}{d:.3f})"

print("=== Overall ===")
print("LaBSE: ", fmt_delta(BASELINE["overall"]["labse"], fo["mean_labse_cosine"]))
print("BLEU:  ", fmt_delta(BASELINE["overall"]["bleu"], fo["corpus_bleu"]))
print("chrF:  ", fmt_delta(BASELINE["overall"]["chrf"], fo["corpus_chrf"]))
print(f"(n={fo['n_sentences']} fine-tuned vs n={BASELINE['overall']['n']} baseline -- same seed/source but re-run separately, so double-check these ns match before trusting the delta as a strict pairing)")

if fp:
    print(f"\n=== Pilot-term-matched subset (n={fp['n_sentences']}, was n={BASELINE['pilot_subset']['n']} in baseline) ===")
    print("LaBSE: ", fmt_delta(BASELINE["pilot_subset"]["labse"], fp["mean_labse_cosine"]))
    print("BLEU:  ", fmt_delta(BASELINE["pilot_subset"]["bleu"], fp["corpus_bleu"]))
else:
    print("\n(no pilot-term matches in this run's subset -- check pilot_term_coverage in the output above)")